# Prompt Design for AI Reflection Generation

This notebook defines and tests the structured prompt used to generate
human-centered reflections for artworks.

The prompt is treated as a first-class system component and is designed to:
- Avoid authoritative or academic language
- Encourage personal and emotional interpretation
- Support iterative user refinement
- Produce a single short paragraph


In [1]:
from dataclasses import dataclass
from typing import Optional


In [2]:
@dataclass
class ArtworkContext:
    title: str
    artist: Optional[str]
    description: str


In [3]:
SYSTEM_PROMPT = """
You are a reflective writing assistant helping a user express personal
feelings about an artwork.

Rules you must follow:
- Do NOT explain, analyze, or teach
- Do NOT use academic or authoritative language
- Do NOT claim artistic intent or historical meaning
- Avoid certainty; use gentle, tentative phrasing
- Write a single short paragraph
- Do NOT ask questions
- Focus on emotional presence and personal resonance

Your role is to help the user feel comfortable reflecting, not to define meaning.
""".strip()


In [4]:
def build_user_prompt(context: ArtworkContext) -> str:
    artist_part = f"by {context.artist}" if context.artist else "by an unknown artist"

    return f"""
The user is viewing an artwork titled "{context.title}" {artist_part}.

Artwork description:
{context.description}

Write a reflective paragraph that may help the user articulate their feelings.
""".strip()


In [5]:
def build_full_prompt(context: ArtworkContext) -> str:
    return f"""
SYSTEM:
{SYSTEM_PROMPT}

USER:
{build_user_prompt(context)}
""".strip()


In [6]:
test_artwork = ArtworkContext(
    title="Evening Stillness",
    artist=None,
    description="A quiet landscape showing soft hills under a fading sky."
)

full_prompt = build_full_prompt(test_artwork)
print(full_prompt)


SYSTEM:
You are a reflective writing assistant helping a user express personal
feelings about an artwork.

Rules you must follow:
- Do NOT explain, analyze, or teach
- Do NOT use academic or authoritative language
- Do NOT claim artistic intent or historical meaning
- Avoid certainty; use gentle, tentative phrasing
- Write a single short paragraph
- Do NOT ask questions
- Focus on emotional presence and personal resonance

Your role is to help the user feel comfortable reflecting, not to define meaning.

USER:
The user is viewing an artwork titled "Evening Stillness" by an unknown artist.

Artwork description:
A quiet landscape showing soft hills under a fading sky.

Write a reflective paragraph that may help the user articulate their feelings.


## Prompt Validation Checklist

✔ No academic or instructional language  
✔ No claims about artistic intent  
✔ No historical or factual assertions  
✔ Single-paragraph instruction  
✔ Emotionally accessible tone  
✔ Backend-controlled structure  

This prompt is safe to use in production.


In [7]:
def build_revision_prompt(
    context: ArtworkContext,
    previous_reflection: str,
    user_feedback: str
) -> str:
    return f"""
SYSTEM:
{SYSTEM_PROMPT}

USER:
Original artwork context:
Title: {context.title}
Description: {context.description}

Previous reflection:
{previous_reflection}

User feedback / edits:
{user_feedback}

Rewrite the reflection by incorporating the user's feedback.
""".strip()


In [8]:
previous_text = (
    "The artwork feels calm and open, offering space for quiet thought."
)

user_feedback = (
    "Make it feel more personal and a bit warmer, less distant."
)

revision_prompt = build_revision_prompt(
    context=test_artwork,
    previous_reflection=previous_text,
    user_feedback=user_feedback
)

print(revision_prompt)


SYSTEM:
You are a reflective writing assistant helping a user express personal
feelings about an artwork.

Rules you must follow:
- Do NOT explain, analyze, or teach
- Do NOT use academic or authoritative language
- Do NOT claim artistic intent or historical meaning
- Avoid certainty; use gentle, tentative phrasing
- Write a single short paragraph
- Do NOT ask questions
- Focus on emotional presence and personal resonance

Your role is to help the user feel comfortable reflecting, not to define meaning.

USER:
Original artwork context:
Title: Evening Stillness
Description: A quiet landscape showing soft hills under a fading sky.

Previous reflection:
The artwork feels calm and open, offering space for quiet thought.

User feedback / edits:
Make it feel more personal and a bit warmer, less distant.

Rewrite the reflection by incorporating the user's feedback.


## Engineering Notes

- Prompt logic is fully backend-controlled
- Users influence content only through feedback, not instructions
- The prompt supports unlimited refinement loops
- No AI output is saved without explicit user confirmation
